In [21]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

In [22]:
# clean col names
cols = {
    "Proposal text": "proposal_text",
    "Status": "status",
    "Is it a housebulding application?": "is_housebuilding",
    "Number of homes this application relates to the creation of": "n_homes",
}

# read xlsx, skip first two rows and only the cols above, then rename them
df = pd.read_excel("data/planning.xlsx", skiprows=2, usecols=list(cols)).rename(
    columns=cols
)
df.columns

Index(['proposal_text', 'status', 'is_housebuilding', 'n_homes'], dtype='str')

In [ ]:
# make sure number of homes is integer, where text stored in this col just drop it
# NOT USED in remaining cells at moment, will be once classify number of houses
df["n_homes"] = (
    pd.to_numeric(df["n_homes"], errors="coerce").astype("Int64").astype("string")
)

In [24]:
# convert all text across all cols to lower case
text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.lower())

In [25]:
# just get the rows that have been checked manually
df_checked = df[df["status"].str.contains("checked", regex=False, na=False)].copy()

In [26]:
# create a binary ml class col for the classifications using the house building flag
# Note: Any yes by reviewer is a positive, so:
# yes but renewal
# yes but change of use
# yes but amendment
# yes but app withdrawn ...etc
df_checked["ml_class"] = (
    df_checked["is_housebuilding"]
    .str.contains("yes", regex=False, na=False)
    .astype(int)
)

In [27]:
# how many rows of each class?
df_checked["ml_class"].value_counts()

ml_class
0    709
1    580
Name: count, dtype: int64

In [28]:

def build_pipeline():
    """
    Setup the model pipeline
    word and char tf-idf features and logistic regression classifier
    """
    word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
    char_tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2)
    features = FeatureUnion([("word", word_tfidf), ("char", char_tfidf)])
    classifier = LogisticRegression(C=10, max_iter=2000)

    return Pipeline([("features", features), ("classifier", classifier)])

In [ ]:
# testing shows we can make this much lower than 0.5 if prioritising recall over precision
CLASSIFICATION_THRESHOLD = 0.25

In [30]:
# cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# run the 5-fold cross validation, refitting the pipeline on each fold's training split
precision_scores = []
recall_scores = []
f1_scores = []
for fold, (train_idx, test_idx) in enumerate(
    skf.split(df_checked["proposal_text"], df_checked["ml_class"]), start=1
):
    X_train, X_test = (
        df_checked["proposal_text"].iloc[train_idx],
        df_checked["proposal_text"].iloc[test_idx],
    )
    y_train, y_test = (
        df_checked["ml_class"].iloc[train_idx],
        df_checked["ml_class"].iloc[test_idx],
    )

    fold_model = build_pipeline()
    fold_model.fit(X_train, y_train)
    y_prob = fold_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= CLASSIFICATION_THRESHOLD).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    print(
        f"Fold {fold} - Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}"
    )

print(f"\nMean Precision: {sum(precision_scores) / len(precision_scores):.4f}")
print(f"Mean Recall: {sum(recall_scores) / len(recall_scores):.4f}")
print(f"Mean F1: {sum(f1_scores) / len(f1_scores):.4f}")

Fold 1 - Precision: 0.8485, Recall: 0.9655, F1: 0.9032
Fold 2 - Precision: 0.8889, Recall: 0.9655, F1: 0.9256
Fold 3 - Precision: 0.8976, Recall: 0.9828, F1: 0.9383
Fold 4 - Precision: 0.8828, Recall: 0.9741, F1: 0.9262
Fold 5 - Precision: 0.9024, Recall: 0.9569, F1: 0.9289

Mean Precision: 0.8841
Mean Recall: 0.9690
Mean F1: 0.9244


In [31]:
# get cross-validated preds for the whole dataset (each row's pred is when not in training fold)
y_prob = cross_val_predict(
    build_pipeline(),
    df_checked["proposal_text"],
    df_checked["ml_class"],
    cv=skf,
    method="predict_proba",
)[:, 1]

df_checked["pred_prob"] = y_prob
df_checked["pred"] = (y_prob >= CLASSIFICATION_THRESHOLD).astype(int)

errors = df_checked[df_checked["pred"] != df_checked["ml_class"]].sort_values(
    "pred_prob"
)

In [32]:
# See what the pred examples look like
df_checked.head()

,proposal_text,status,is_housebuilding,n_homes,ml_class,pred_prob,pred
2,erection of 1 house and associated works,checked,yes,1,1,0.994318,1
5,substitution of house types on plots 39 to 46 (amendment to 20/03233/arc),checked,yes but s42,7,1,0.887524,1
7,erection of two dwellinghouses and associated works,checked,yes,2,1,0.997658,1
10,erection of dwellinghouse and associated works (demolition of existing garage),checked,yes,1,1,0.969242,1
17,"installation of replacement windows to flats 0/2, 1/1, 2/1 and 2/3.",checked,no,<NA>,0,0.150833,0


In [33]:
# confusion matrix from out-of-fold predictions
cm = confusion_matrix(df_checked["ml_class"], df_checked["pred"])
pd.DataFrame(
    cm,
    index=["actual: not housebuilding", "actual: housebuilding"],
    columns=["pred: not housebuilding", "pred: housebuilding"],
)

,pred: not housebuilding,pred: housebuilding
actual: not housebuilding,635,74
actual: housebuilding,18,562


### Result
18 of 580 labelled housing applications were missed, at the cost of 74 false positives

In [34]:
# write out for manual inspection
df_checked.to_csv("data/predict.csv")

### False positives - examples with strongest +ve prediction
Some of these appear to be labelling errors?

In [35]:
pd.set_option("display.max_colwidth", None)
# show top 10 where most strongly predicted as housing, but not labelled as housing in spreadsheet
df_checked[df_checked["ml_class"] == 0].sort_values("pred_prob", ascending=False)[
    ["proposal_text", "is_housebuilding", "pred_prob"]
].head(10)

,proposal_text,is_housebuilding,pred_prob
4268,demolition of existing dwellinghouse and erection of replacement dwellinghouse and associated infrastructure,no,0.998551
1721,"renewal of planning permission reference 22/01243/pp (demolition of ruin, erection of dwellinghouse and installation of sewage treatment plant)",no,0.962958
1168,conversion of two flats into one house,no,0.962756
6450,convert existing adjoined garage to living accommodation at dwelling house.,no,0.953072
3400,"alteration and extension to outbuilding to rear of dwellinghouse to form new two storey dwellinghouse and erection of single storey extension to the existing dwelling to adjoin the new dwelling, thereby resulting in the formation of a semi-enclosed courtyard between dwellings",no,0.912039
4725,use of 2no. flatted dwellings as 1no. dwellinghouse with erection of single storey extension to rear,no,0.899628
5379,alterations and change of use of dwellinghouses (class 9) to form additional hotel guest accommodation (class 7),no,0.894845
6552,extension to dwellinghouse,no,0.860808
6100,amalgamation of 2no flatted dwellings (ground floor flats) to form 1no flatted dwelling.,no,0.859426
4709,erection of dwelling for agricultural worker and associated parking and landscaping works (renewal of previous planning permission 2022/0033/tp),no,0.825849


### False negative - examples with strongest -ve prediction

In [36]:
# show top 10 where not predicted as housing, but labelled as housing
df_checked[df_checked["ml_class"] == 1].sort_values("pred_prob", ascending=True)[
    ["proposal_text", "is_housebuilding", "pred_prob"]
].head(10)

,proposal_text,is_housebuilding,pred_prob
4776,change the design of consented application 22/01315/app noting a reduced ground floor area and reduced ridge height on plot 1 at,yes but change of spec of plot 1,0.036153
1792,convert upper floor of former care home into 7no flats,"yes, but change of use",0.058142
4572,erection of first floor flat,yes,0.065335
1556,demolition of existing entrance porch; erection of new garage and vestibule; erection of new external balcony from dining room; conversion of former ground floor garage and store to granny flat.,"yes, but change of use",0.080819
6412,"condition 2(a) siting, design, layout, external appearance, materials, waste bin uplift area, driveway, parking, turning area, ev charging point (b) means of access, visibility splays (c) landscaping scheme (d) levels survey, cross sections, finished floor levels and (e) drainage of planning permission reference app/2023/1871 for erection of dwellinghouse and garage",yes,0.094227
1731,change of use from church session rooms to single family home. minor external alterations including installation of 3 no. heritage roof lights and replacement of 2 solid doors to rear with new glazed doors (as amended).,"yes, but change of use",0.118591
6285,alterations and change of use of office accommodation to 2 no. flats,yes,0.131840
6266,section 42 application to develop without compliance with condition 4 of permission 20/01602/ful to allow the self catering unit to be occupied prior to the compeltion fo the house,yes but s42,0.139655
2696,change of use from vacant office to residential apartment with new window and block off existing external doorway.,"yes, but change of use",0.141603
3755,proposed erection of single home for retirement succession.,yes,0.150722
